In [2]:
# --- IMPORTS ---
import os
import pandas as pd
import duckdb

from crewai.llm import LLM
from llama_index.core import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine

# --- API KEY ---
api_key=os.environ["OPENAI_API_KEY"]

# --- LOAD DATA ---
url = "https://drive.google.com/uc?export=download&id=1s4T0-L4-LgoUCj0WNU1JVH2-jHXDiWf6"
df = pd.read_csv(url)

# --- SINGLE CONNECTION (IMPORTANT) ---
con = duckdb.connect(database=":memory:")   # use in-memory to avoid file conflicts

# Register dataframe directly
con.register("products_df", df)

# Create table from dataframe
con.execute("""
    CREATE TABLE products AS
    SELECT * FROM products_df
""")

# --- WRAP CONNECTION FOR LLAMAINDEX ---
# KEY: use creator to avoid new connections
from sqlalchemy import create_engine

engine = create_engine(
    "duckdb:///:memory:",
    creator=lambda: con
)

# --- CREATE SQL DATABASE (NO REFLECTION BUG) ---
sql_db = SQLDatabase(
    engine,
    include_tables=["products"]   # limit scope
)

# --- LLM ---
llm = LLM(model="gpt-4o-mini")

# --- QUERY ENGINE ---
query_engine = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=["products"],
    llm=llm
)

# --- TEST QUERY ---
response = query_engine.query("What are the top 5 most expensive products?")
print(response)

AttributeError: '_duckdb.DuckDBPyConnection' object has no attribute 'connection'